<a href="https://colab.research.google.com/github/Froznrobt/sudo-proxy/blob/dev/experiment4_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip instal biopython

ERROR: unknown command "instal" - maybe you meant "install"


In [ ]:
"""
Restriction mapping + AT-palindrome/overhang search + cut site visualization + primer design
Works in Google Colab (upload dialog) or locally (python script.py file.fasta)
"""
import re
import sys

# ---------- 1. Read FASTA or raw sequence file ----------
def read_fasta(filename):
    """Returns list of (header, sequence) tuples."""
    records, header, chunks = [], None, []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(chunks).upper()))
                header, chunks = line[1:], []
            else:
                if header is None:
                    header = "unnamed_sequence"
                chunks.append(line)
    if header is not None:
        records.append((header, "".join(chunks).upper()))
    if not records:
        raise ValueError("No valid DNA sequence found in file: " + filename)

    cleaned = []
    for h, sq in records:
        clean_sq = re.sub(r"[^ACGT]", "", sq)
        cleaned.append((h, clean_sq))
    return cleaned


# ---------- 2. Basic helpers ----------
COMP = str.maketrans("ACGT", "TGCA")

def revcomp(s):
    return s.translate(COMP)[::-1]

def gc_content(s):
    return 100 * (s.count("G") + s.count("C")) / len(s) if s else 0.0

def at_content(s):
    return 100 - gc_content(s)

def is_palindrome(s):
    return s == revcomp(s)

def tm(s):
    """Wallace rule for <14 nt, else basic GC formula."""
    if not s:
        return 0.0
    gc, at = s.count("G") + s.count("C"), s.count("A") + s.count("T")
    return 2 * at + 4 * gc if len(s) < 14 else 64.9 + 41 * (gc - 16.4) / len(s)


# ---------- 3. Palindromic Sequence Finder ----------
def find_palindromes(seq, min_len=4, max_len=8):
    """A DNA palindrome = sequence equal to its own reverse complement (even length only)."""
    hits = []
    for L in range(min_len, max_len + 1, 2):
        for i in range(len(seq) - L + 1):
            sub = seq[i:i + L]
            if sub == revcomp(sub):
                hits.append((i + 1, L, sub))  # 1-based position
    return hits


# ---------- 4. Restriction Enzyme Database & Overhang Calculation ----------
ENZYMES = {
    "EcoRI":   ("GAATTC", 1), "BamHI":  ("GGATCC", 1), "HindIII": ("AAGCTT", 1),
    "KpnI":    ("GGTACC", 5), "SacI":   ("GAGCTC", 5), "XbaI":    ("TCTAGA", 1),
    "XhoI":    ("CTCGAG", 1), "PstI":   ("CTGCAG", 5), "SalI":    ("GTCGAC", 1),
    "NcoI":    ("CCATGG", 1), "NdeI":   ("CATATG", 2), "NotI":    ("GCGGCCGC", 2),
    "SmaI":    ("CCCGGG", 3), "EcoRV":  ("GATATC", 3), "ClaI":    ("ATCGAT", 2),
    "SpeI":    ("ACTAGT", 1), "NheI":   ("GCTAGC", 1), "ApaI":    ("GGGCCC", 5),
    "AluI":    ("AGCT", 2),   "HaeIII": ("GGCC", 2),   "TaqI":    ("TCGA", 1),
    "MspI":    ("CCGG", 1),   "BglII":  ("AGATCT", 1), "SphI":    ("GCATGC", 5),
    # --- Enzymes whose recognition sites are strictly A and T ---
    "MseI":    ("TTAA", 1),   "MluCI":  ("AATT", 0),   "DraI":    ("TTTAAA", 3),
    "SspI":    ("AATATT", 3), "SwaI":   ("ATTTAAAT", 4), "AseI":  ("ATTAAT", 2),
    "PacI":    ("TTAATTAA", 5),
}

AT_ONLY_ENZYMES = {n: v for n, v in ENZYMES.items() if set(v[0]) <= set("AT")}


def cohesive_end(site, cut):
    """Calculates end type, overhang length, and 5'->3' overhang sequence on top strand."""
    L, b = len(site), len(site) - cut
    if cut == b:
        return "blunt", 0, "-"
    if cut < b:
        return "5' overhang (sticky)", b - cut, site[cut:b]
    return "3' overhang (sticky)", cut - b, site[b:cut]


def draw_cut(site, cut):
    """Generates a two-strand visualization of the cleavage pattern."""
    b = len(site) - cut
    comp = site.translate(COMP)
    top = site[:cut] + "  |  " + site[cut:]
    bot = comp[:b] + "  |  " + comp[b:]
    return f"5'-{top}-3'\n3'-{bot}-5'"


def find_sites(seq, site):
    return [m.start() for m in re.finditer(f"(?=({site}))", seq)]


def restriction_map(seq):
    result = {}
    for name, (site, cut) in ENZYMES.items():
        pos = find_sites(seq, site)
        if pos:
            cuts = [p + cut for p in pos]
            edges = sorted(list(set([0] + cuts + [len(seq)])))
            frags = [edges[i + 1] - edges[i] for i in range(len(edges) - 1)]
            result[name] = dict(site=site, sites=[p + 1 for p in pos], cuts=cuts, fragments=frags)
    return result


def display_sequence_cut_sites(seq, rmap):
    """Shows exact 1-based cut positions and local sequence context around each cleavage point."""
    print(f"{'Enzyme':<10}{'Recognition Site':<18}{'Cut BP Location':<18}{'Local Sequence Context (5\'->3\')'}")
    print("-" * 75)
    for name, data in sorted(rmap.items(), key=lambda x: x[1]["cuts"][0]):
        site = data["site"]
        for cut_pos in data["cuts"]:
            # Context window: 5 bases left and right of cut position
            left_flank = seq[max(0, cut_pos - 5):cut_pos]
            right_flank = seq[cut_pos:min(len(seq), cut_pos + 5)]
            context = f"... {left_flank} | {right_flank} ..."
            print(f"{name:<10}{site:<18}{cut_pos:<18}{context}")


# ---------- 5. A/T-Specific Palindrome & Overhang Analyzer ----------
def analyze_at_palindromes(seq, min_len=4, max_len=8):
    """Identifies pure A/T palindromes, reports AT%, and checks overhang properties."""
    all_pals = find_palindromes(seq, min_len, max_len)
    at_pals = [p for p in all_pals if set(p[2]) <= set("AT")]

    results = []
    for pos, length, pal in at_pals:
        matching_enzymes = [n for n, (st, c) in AT_ONLY_ENZYMES.items() if st == pal]

        enz_info = []
        for name in matching_enzymes:
            st, cut = AT_ONLY_ENZYMES[name]
            kind, ov_len, ov_seq = cohesive_end(st, cut)
            enz_info.append({
                "enzyme": name,
                "type": kind,
                "overhang_length": ov_len,
                "overhang_seq": ov_seq
            })

        results.append({
            "position": pos,
            "length": length,
            "sequence": pal,
            "at_percent": at_content(pal),
            "enzymes": enz_info
        })

    return results


# ---------- 6. Primer design ----------
def design_primer(template, length_range=(18, 25), tm_range=(50, 65)):
    """Pick shortest primer with GC 40-60%, Tm in range, and GC clamp."""
    min_len, max_len = length_range[0], length_range[1]
    for L in range(min_len, max_len + 1):
        p = template[:L]
        if len(p) < min_len:
            continue
        if 40 <= gc_content(p) <= 60 and tm_range[0] <= tm(p) <= tm_range[1] and p[-1] in "GC":
            return p
    return template[:20]


def design_primers(seq, start=0, end=None):
    end = end or len(seq)
    region = seq[start:end]
    fwd = design_primer(region)
    rev = design_primer(revcomp(region))
    return fwd, rev


# ---------- 7. Main Analysis Output ----------
def analyze(desc, seq):
    print('\n' + '=' * 75 + '\n>' + desc)
    print("Sequence Length: %d bp | Global GC: %.1f%% | Global AT: %.1f%%" % (len(seq), gc_content(seq), at_content(seq)))
    print("Nucleotide Counts: A=%d T=%d G=%d C=%d" % tuple(seq.count(b) for b in "ATGC"))

    print("\n" + "-" * 25 + " EXACT ENZYME CUT SITES IN SEQUENCE " + "-" * 25)
    rmap = restriction_map(seq)
    if rmap:
        display_sequence_cut_sites(seq, rmap)
    else:
        print("No matching restriction enzyme cut sites found in sequence.")

    print("\n" + "-" * 25 + " A/T-ONLY PALINDROMES & OVERHANGS " + "-" * 25)
    at_results = analyze_at_palindromes(seq)
    print(f"Total pure A/T palindromes found (4-8 bp): {len(at_results)}")

    if at_results:
        print(f"\n{'Pos (1-based)':<15}{'Seq':<12}{'AT %':<8}{'Matched Enzyme':<16}{'Overhang Type':<22}{'Overhang Seq'}")
        print("-" * 80)
        for r in at_results[:20]:
            if r["enzymes"]:
                for enz in r["enzymes"]:
                    print(f"{r['position']:<15}{r['sequence']:<12}{r['at_percent']:<8.1f}{enz['enzyme']:<16}{enz['type']:<22}{enz['overhang_seq']}")
            else:
                print(f"{r['position']:<15}{r['sequence']:<12}{r['at_percent']:<8.1f}{'No match':<16}{'-':<22}{'-'}")

    print("\n" + "-" * 25 + " RESTRICTION MAP & FRAGMENT SIZES " + "-" * 25)
    for name, d in sorted(rmap.items(), key=lambda x: x[1]["cuts"][0]):
        print(f"{name:8s} {d['site']:8s} recognition_starts={d['sites']}  cut_after_bp={d['cuts']}  fragments={d['fragments']}")

    print("\n" + "-" * 25 + " CLEAVAGE PATTERNS & OVERHANGS " + "-" * 25)
    for name in rmap:
        site, cut = ENZYMES[name]
        kind, n, oh = cohesive_end(site, cut)
        print(f"{name} ({site}): {kind} | AT%={at_content(site):.1f}% | Overhang={n} nt [{oh}]")
        print(draw_cut(site, cut), "\n")

    print("\n" + "-" * 25 + " PRIMER DESIGN " + "-" * 25)
    f, r = design_primers(seq)
    for label, p in (("Forward 5'->3'", f), ("Reverse 5'->3'", r)):
        print(f"{label}: {p}  len={len(p)}  GC={gc_content(p):.1f}%  Tm={tm(p):.1f}C")


def get_fasta_path():
    import os
    args = [a for a in sys.argv[1:] if not a.startswith("-") and os.path.isfile(a)]
    if args:
        return args[0]
    try:
        from google.colab import files
        print("Upload your FASTA file (.fasta / .fa / .txt):")
        return next(iter(files.upload()))
    except ImportError:
        return input("Enter path to FASTA file: ").strip()


if __name__ == "__main__":
    fname = get_fasta_path()
    for desc, seq in read_fasta(fname):
        analyze(desc, seq)


>unnamed_sequence
Sequence Length: 31 bp | Global GC: 32.3% | Global AT: 67.7%
Nucleotide Counts: A=8 T=13 G=1 C=9

------------------------- EXACT ENZYME CUT SITES IN SEQUENCE -------------------------
Enzyme    Recognition Site  Cut BP Location   Local Sequence Context (5'->3')
---------------------------------------------------------------------------
MluCI     AATT              10                ... TTTTC | AATTC ...
ClaI      ATCGAT            22                ... CCTAT | CGATC ...
TaqI      TCGA              22                ... CCTAT | CGATC ...

------------------------- A/T-ONLY PALINDROMES & OVERHANGS -------------------------
Total pure A/T palindromes found (4-8 bp): 1

Pos (1-based)  Seq         AT %    Matched Enzyme  Overhang Type         Overhang Seq
--------------------------------------------------------------------------------
11             AATT        100.0   MluCI           5' overhang (sticky)  AATT

------------------------- RESTRICTION MAP & FRAGMENT SIZES -